# Dense Passage Retrieval (DPR) Evaluation Pipeline — Local Indexes & CLI Commands

Uses **already-downloaded local Pyserini indexes** (`C:\Users\Dwija Patel\.cache\pyserini\indexes\`) without re-downloading, and executes `sys.executable` (`{py}`) CLI commands (`pyserini.search.lucene`, `pyserini.eval.convert_trec_run_to_dpr_retrieval_run`, `pyserini.eval.evaluate_dpr_retrieval`) across:
- **All 5 datasets (`nq`, `trivia`, `wq`, `curated`, `squad`)** on the **Multi-dataset index** (`faiss.wikipedia-dpr-100w.dpr_multi.*`) and **BM25** (`lucene-inverted.wikipedia-dpr-100w.*`).
- **NQ only (`nq`)** on the **NQ-only index** (`faiss.wikipedia-dpr-100w.dpr_single-nq.*`).

## 1. Imports & Local Index Auto-Discovery 

In [12]:
import os
import re
import sys
import json
import struct
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer
from pyserini.search import get_topics

py = sys.executable
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

RUNS_DIR = Path('runs')
CKPT_DIR = Path('checkpoints')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve your existing local Pyserini index cache directly
INDEX_CACHE_DIR = Path(os.path.expanduser('~')) / '.cache' / 'pyserini' / 'indexes'

def find_local_index(prefix: str, cache_dir: Path = INDEX_CACHE_DIR) -> str:
    matches = sorted([p for p in cache_dir.glob(f'{prefix}*') if p.is_dir()])
    if not matches:
        raise FileNotFoundError(f'Could not find local index matching {prefix}* inside {cache_dir}')
    return str(matches[-1])

root = find_local_index('lucene-inverted.wikipedia-dpr-100w')
faiss_multi_root = find_local_index('faiss.wikipedia-dpr-100w.dpr_multi')
faiss_single_nq_root = find_local_index('faiss.wikipedia-dpr-100w.dpr_single-nq')

print('Python executable (py):', py)
print('Device:', DEVICE)
print('Lucene root:', root)
print('FAISS Multi root:', faiss_multi_root)
print('FAISS Single-NQ root:', faiss_single_nq_root)

Python executable (py): c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe
Device: cpu
Lucene root: C:\Users\Dwija Patel\.cache\pyserini\indexes\lucene-inverted.wikipedia-dpr-100w.20260508.deb4c7b
FAISS Multi root: C:\Users\Dwija Patel\.cache\pyserini\indexes\faiss.wikipedia-dpr-100w.dpr_multi.20200127.f403c3
FAISS Single-NQ root: C:\Users\Dwija Patel\.cache\pyserini\indexes\faiss.wikipedia-dpr-100w.dpr_single-nq.20200115.cd5034


## 2. Dataset & Index Routing Configuration

- **Index built on all datasets (`faiss_multi_root`)** $\rightarrow$ evaluated on **all 5 datasets** (`nq`, `trivia`, `wq`, `curated`, `squad`).
- **Index built only on NQ (`faiss_single_nq_root`)** $\rightarrow$ evaluated **strictly on NQ** (`nq`).

In [13]:
ALL_DATASETS = {
    'nq':      {'topics': 'dpr-nq-test',      'tag': 'nq-test',      'regex': False},
    'trivia':  {'topics': 'dpr-trivia-test',  'tag': 'trivia-test',  'regex': False},
    'wq':      {'topics': 'dpr-wq-test',      'tag': 'wq-test',      'regex': False},
    'curated': {'topics': 'dpr-curated-test', 'tag': 'curated-test', 'regex': True},
    'squad':   {'topics': 'dpr-squad-test',   'tag': 'squad-test',   'regex': False},
}

NQ_ONLY_DATASETS = {
    'nq': ALL_DATASETS['nq']
}

DENSE_CONFIGS = {
    'multi': {
        'local_index_dir': faiss_multi_root,
        'encoder_name': 'facebook/dpr-question_encoder-multiset-base',
        'datasets': ALL_DATASETS,      # All datasets index -> run on ALL datasets (incl. SQuAD)
        'alpha': 1.1,
    },
    'dpr-single-nq': {
        'local_index_dir': faiss_single_nq_root,
        'encoder_name': 'facebook/dpr-question_encoder-single-nq-base',
        'datasets': NQ_ONLY_DATASETS,  # NQ-only index -> run ONLY on NQ
        'alpha': 1.1,
    },
}

TOP_K_RETRIEVE = 1000
CHUNK_SIZE = 50_000

## 3. Exact Pyserini CLI Conversion & Evaluation Helper (`-X utf8`)

In [14]:
def run_cmd(cmd_list):
    print('>>', ' '.join(str(x) for x in cmd_list))
    env = {**os.environ, 'PYTHONUTF8': '1'}
    proc = subprocess.run(cmd_list, capture_output=True, text=True, encoding='utf-8', errors='replace', env=env)
    combined = (proc.stdout or '') + '\n' + (proc.stderr or '')
    if combined.strip():
        print(combined.strip())
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd_list, output=proc.stdout, stderr=proc.stderr)
    return combined


def convert_and_eval_trec(trec_path: Path, topics_name: str, use_regex: bool = False):
    """Runs convert_trec_run_to_dpr_retrieval_run and evaluate_dpr_retrieval with UTF-8 forced on Windows."""
    json_path = trec_path.with_suffix('.json')

    if (not json_path.exists()) or json_path.stat().st_mtime < trec_path.stat().st_mtime:
        convert_cmd = [
            py, '-X', 'utf8', '-m', 'pyserini.eval.convert_trec_run_to_dpr_retrieval_run',
            '--index', root,
            '--topics', topics_name,
            '--input', str(trec_path),
            '--output', str(json_path),
        ]
        if use_regex:
            convert_cmd.append('--regex')
        run_cmd(convert_cmd)

    eval_cmd = [
        py, '-X', 'utf8', '-m', 'pyserini.eval.evaluate_dpr_retrieval',
        '--retrieval', str(json_path),
        '--topk', '20', '100',
    ]
    if use_regex:
        eval_cmd.append('--regex')

    out_text = run_cmd(eval_cmd)
    m20 = re.search(r'Top20\s+accuracy:\s*([0-9.]+)', out_text)
    m100 = re.search(r'Top100\s+accuracy:\s*([0-9.]+)', out_text)
    if not m20 or not m100:
        raise RuntimeError(f'Failed to parse Top20/Top100 accuracy from output:\n{out_text}')

    return {
        'Top-20': round(float(m20.group(1)) * 100, 2),
        'Top-100': round(float(m100.group(1)) * 100, 2),
    }

## 4. Run BM25 Across All 5 Datasets Using `pyserini.search.lucene` & Local `root`

In [15]:
bm25_records = []

for ds_key, ds_cfg in ALL_DATASETS.items():
    topics_name = ds_cfg['topics']
    tag = ds_cfg['tag']
    trec_out = RUNS_DIR / f'run.dpr.{tag}.bm25.trec'

    if not trec_out.exists():
        run_cmd([
            py, '-m', 'pyserini.search.lucene',
            '--index', root,
            '--topics', topics_name,
            '--output', str(trec_out),
            '--hits', str(TOP_K_RETRIEVE),
            '--threads', '8',
            '--batch-size', '128',
            '--bm25',
        ])
    else:
        print(f'[Skip] Existing BM25 run found: {trec_out}')

    scores = convert_and_eval_trec(trec_out, topics_name, use_regex=ds_cfg['regex'])
    bm25_records.append({
        'Dataset': ds_key.upper(),
        'Retriever': 'BM25',
        'Index': Path(root).name,
        'Top-20': scores['Top-20'],
        'Top-100': scores['Top-100'],
    })

display(pd.DataFrame(bm25_records))

[Skip] Existing BM25 run found: runs\run.dpr.nq-test.bm25.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.nq-test.bm25.json --topk 20 100
Top20	accuracy: 0.6294
Top100	accuracy: 0.7825


100%|██████████| 3610/3610 [00:00<00:00, 196621.62it/s]
[Skip] Existing BM25 run found: runs\run.dpr.trivia-test.bm25.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.trivia-test.bm25.json --topk 20 100
Top20	accuracy: 0.7641
Top100	accuracy: 0.8315


100%|██████████| 11313/11313 [00:00<00:00, 288979.60it/s]
[Skip] Existing BM25 run found: runs\run.dpr.wq-test.bm25.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.wq-test.bm25.json --topk 20 100
Top20	accuracy: 0.6240
Top100	accuracy: 0.7549


100%|██████████| 

,Dataset,Retriever,Index,Top-20,Top-100
0,NQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.94,78.25
1,TRIVIA,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,76.41,83.15
2,WQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.40,75.49
3,CURATED,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,80.69,89.91
4,SQUAD,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,71.09,81.84


## 5. Low-RAM `np.memmap` Chunked Exact Inner-Product Search on Local FAISS Indexes

In [16]:
def open_local_faiss_memmap(local_index_dir: str):
    """Memory-maps your local FAISS IndexFlatIP file without loading 64 GB into RAM."""
    index_path = Path(local_index_dir)
    candidates = sorted(index_path.glob('*.faiss')) or [
        p for p in index_path.iterdir() if p.is_file() and p.name not in {'docid', 'README.md'}
    ]
    if not candidates:
        raise FileNotFoundError(f'No FAISS binary found inside {local_index_dir}')
    faiss_file = max(candidates, key=lambda p: p.stat().st_size)

    with open(faiss_file, 'rb') as f:
        fourcc = f.read(4)
        if fourcc != b'IxFI':
            raise ValueError(f'Expected IndexFlatIP (IxFI), got {fourcc} in {faiss_file}')
        dim = struct.unpack('<i', f.read(4))[0]
        ntotal = struct.unpack('<q', f.read(8))[0]
        _dummy1 = struct.unpack('<q', f.read(8))[0]
        _dummy2 = struct.unpack('<q', f.read(8))[0]
        _is_trained = struct.unpack('<?', f.read(1))[0]
        _metric_type = struct.unpack('<i', f.read(4))[0]
        vec_count = struct.unpack('<Q', f.read(8))[0]
        data_offset = f.tell()

    assert vec_count == ntotal * dim, f'Header mismatch: {vec_count} != {ntotal}*{dim}'

    with open(index_path / 'docid', 'r', encoding='utf-8') as f:
        docids = [line.strip() for line in f]
    assert len(docids) == ntotal, f'Docid count ({len(docids)}) != ntotal ({ntotal})'

    vectors = np.memmap(
        faiss_file, dtype=np.float32, mode='r', offset=data_offset, shape=(ntotal, dim)
    )
    print(f'Memory-mapped local index {faiss_file.name}: shape={vectors.shape}')
    return vectors, docids


@torch.no_grad()
def encode_queries(topics_dict: dict, encoder_name: str, batch_size: int = 64):
    tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(encoder_name)
    model = DPRQuestionEncoder.from_pretrained(encoder_name).to(DEVICE)
    model.eval()

    qids = list(topics_dict.keys())
    queries = [topics_dict[qid]['title'] for qid in qids]
    all_embs = []

    for start in range(0, len(queries), batch_size):
        batch_q = queries[start:start + batch_size]
        inputs = tokenizer(
            batch_q, padding=True, truncation=True, max_length=256, return_tensors='pt'
        ).to(DEVICE)
        emb = model(**inputs).pooler_output.cpu().numpy().astype(np.float32)
        all_embs.append(emb)

    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return qids, np.vstack(all_embs)


def chunked_exact_ip_search(
    passage_memmap: np.memmap,
    docids: list,
    qids: list,
    query_embs: np.ndarray,
    trec_out_path: Path,
    ckpt_path: Path,
    run_tag: str,
    topk: int = 1000,
    chunk_size: int = 50_000,
    ckpt_every_chunks: int = 20,
):
    if trec_out_path.exists():
        print(f'[Skip] Existing TREC run found: {trec_out_path}')
        return

    ntotal = passage_memmap.shape[0]
    num_q = len(qids)
    start_row = 0

    if ckpt_path.exists():
        ckpt = np.load(ckpt_path)
        start_row = int(ckpt['next_row'])
        best_scores = ckpt['best_scores']
        best_indices = ckpt['best_indices']
        print(f'Resuming {run_tag} from row {start_row:,}/{ntotal:,}')
    else:
        best_scores = np.full((num_q, topk), -np.inf, dtype=np.float32)
        best_indices = np.full((num_q, topk), -1, dtype=np.int64)

    q_T = np.ascontiguousarray(query_embs.T)
    total_chunks = (ntotal - start_row + chunk_size - 1) // chunk_size

    for chunk_idx, row_start in enumerate(
        tqdm(range(start_row, ntotal, chunk_size), total=total_chunks, desc=run_tag), start=1
    ):
        row_end = min(row_start + chunk_size, ntotal)
        chunk = np.asarray(passage_memmap[row_start:row_end], dtype=np.float32)
        scores_T = (chunk @ q_T).T

        k_chunk = min(topk, row_end - row_start)
        part_idx = np.argpartition(-scores_T, kth=k_chunk - 1, axis=1)[:, :k_chunk]
        part_scores = np.take_along_axis(scores_T, part_idx, axis=1)
        global_idx = part_idx + row_start

        merged_scores = np.concatenate([best_scores, part_scores], axis=1)
        merged_indices = np.concatenate([best_indices, global_idx], axis=1)

        top_merge = np.argpartition(-merged_scores, kth=topk - 1, axis=1)[:, :topk]
        best_scores = np.take_along_axis(merged_scores, top_merge, axis=1)
        best_indices = np.take_along_axis(merged_indices, top_merge, axis=1)

        if chunk_idx % ckpt_every_chunks == 0:
            np.savez(
                ckpt_path,
                next_row=np.array(row_end, dtype=np.int64),
                best_scores=best_scores,
                best_indices=best_indices,
            )

    order = np.argsort(-best_scores, axis=1)
    sorted_scores = np.take_along_axis(best_scores, order, axis=1)
    sorted_indices = np.take_along_axis(best_indices, order, axis=1)

    with open(trec_out_path, 'w', encoding='utf-8') as f:
        for qi, qid in enumerate(qids):
            for rank in range(topk):
                doc_idx = int(sorted_indices[qi, rank])
                if doc_idx < 0:
                    continue
                f.write(f'{qid} Q0 {docids[doc_idx]} {rank + 1} {sorted_scores[qi, rank]:.6f} {run_tag}\n')

    if ckpt_path.exists():
        ckpt_path.unlink()
    print(f'[Done] Wrote TREC run: {trec_out_path}')


def load_trec_scores(trec_path: Path):
    runs = {}
    with open(trec_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                qid, _, docid, _, score, _ = parts[:6]
                runs.setdefault(qid, {})[docid] = float(score)
    return runs


def build_hybrid_trec_run(dense_trec: Path, bm25_trec: Path, hybrid_out: Path, alpha: float, run_tag: str, topk: int = 1000):
    if hybrid_out.exists():
        print(f'[Skip] Existing Hybrid run found: {hybrid_out}')
        return
    dense_runs = load_trec_scores(dense_trec)
    bm25_runs = load_trec_scores(bm25_trec)
    with open(hybrid_out, 'w', encoding='utf-8') as f:
        for qid, d_docs in dense_runs.items():
            b_docs = bm25_runs.get(qid, {})
            d_min = min(d_docs.values())                       # missing dense score -> min of dense list
            b_min = min(b_docs.values()) if b_docs else 0.0    # missing BM25 score -> min of BM25 list
            scored = [(x, b_docs.get(x, b_min) + alpha * d_docs.get(x, d_min))
                for x in set(d_docs) | set(b_docs)]
            scored.sort(key=lambda t: t[1], reverse=True)
            for rank, (docid, score) in enumerate(scored[:topk], start=1):
                f.write(f'{qid} Q0 {docid} {rank} {score:.6f} {run_tag}\n')
    print(f'[Done] Wrote Hybrid TREC run: {hybrid_out}')

## 6. Run Dense & Hybrid Retrieval + CLI Evaluation

- **`dpr-multi` (`faiss_multi_root`)** runs across **all 5 datasets** (NQ, TriviaQA, WebQuestions, CuratedTREC, **SQuAD**).
- **`dpr-single-nq` (`faiss_single_nq_root`)** runs **only on NQ** (`dpr-nq-test`).

In [17]:
from concurrent.futures import ThreadPoolExecutor

MAX_PARALLEL = 3   # start with 3; raise to 6 if RAM and disk allow

# ---------- Phase 1: dense runs (only if missing) + hybrid TREC files (fast, sequential) ----------
jobs = []  # one entry per (retriever, dataset) that needs evaluation

for mode_key, cfg in DENSE_CONFIGS.items():
    print(f'\n=== {mode_key} ===')
    passage_memmap = docids = None

    for ds_key, ds_cfg in cfg['datasets'].items():
        tag, topics_name, use_reg = ds_cfg['tag'], ds_cfg['topics'], ds_cfg['regex']
        dense_trec = RUNS_DIR / f'run.dpr.{tag}.{mode_key}.trec'
        ckpt_file  = CKPT_DIR / f'ckpt.dpr.{tag}.{mode_key}.npz'

        if dense_trec.exists():
            print(f'[Skip] Existing TREC run found: {dense_trec}')
        else:
            if passage_memmap is None:
                passage_memmap, docids = open_local_faiss_memmap(cfg['local_index_dir'])
            topics = get_topics(topics_name)
            qids, query_embs = encode_queries(topics, cfg['encoder_name'])
            chunked_exact_ip_search(
                passage_memmap=passage_memmap, docids=docids, qids=qids, query_embs=query_embs,
                trec_out_path=dense_trec, ckpt_path=ckpt_file, run_tag=mode_key,
                topk=TOP_K_RETRIEVE, chunk_size=CHUNK_SIZE,
            )

        bm25_trec  = RUNS_DIR / f'run.dpr.{tag}.bm25.trec'
        hybrid_tag = 'hybrid-multi' if mode_key == 'multi' else 'hybrid-single-nq'
        hybrid_trec = RUNS_DIR / f'run.hybrid.{tag}.{hybrid_tag}.trec'
        build_hybrid_trec_run(
            dense_trec=dense_trec, bm25_trec=bm25_trec, hybrid_out=hybrid_trec,
            alpha=cfg['alpha'], run_tag=hybrid_tag, topk=100,
        )

        dense_name  = 'DPR-Multi' if mode_key == 'multi' else 'DPR-Single (NQ)'
        hybrid_name = 'BM25 + DPR-Multi' if mode_key == 'multi' else 'BM25 + DPR-Single (NQ)'
        idx_name = Path(cfg['local_index_dir']).name
        jobs.append(dict(ds=ds_key.upper(), name=dense_name,  index=idx_name, trec=dense_trec,
                         topics=topics_name, regex=use_reg))
        jobs.append(dict(ds=ds_key.upper(), name=hybrid_name, index=idx_name, trec=hybrid_trec,
                         topics=topics_name, regex=use_reg))

    del passage_memmap

# ---------- Phase 2: convert + evaluate everything in parallel ----------
def run_job(j):
    scores = convert_and_eval_trec(j['trec'], j['topics'], use_regex=j['regex'])
    return {'Dataset': j['ds'], 'Retriever': j['name'], 'Index': j['index'],
            'Top-20': scores['Top-20'], 'Top-100': scores['Top-100']}

print(f'\nRunning {len(jobs)} convert+evaluate jobs with {MAX_PARALLEL} in parallel...')
with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
    job_records = list(ex.map(run_job, jobs))   # keeps the original order

# ---------- Phase 3: collect results ----------
all_records = list(bm25_records) + job_records
results_df = pd.DataFrame(all_records)
results_df.to_csv(RUNS_DIR / 'dpr_final_table_results.csv', index=False)
pivot_df = results_df.pivot(index='Retriever', columns='Dataset', values=['Top-20', 'Top-100'])
display(results_df)
display(pivot_df)


=== multi ===
[Skip] Existing TREC run found: runs\run.dpr.nq-test.multi.trec
[Skip] Existing Hybrid run found: runs\run.hybrid.nq-test.hybrid-multi.trec
[Skip] Existing TREC run found: runs\run.dpr.trivia-test.multi.trec
[Skip] Existing Hybrid run found: runs\run.hybrid.trivia-test.hybrid-multi.trec
[Skip] Existing TREC run found: runs\run.dpr.wq-test.multi.trec
[Skip] Existing Hybrid run found: runs\run.hybrid.wq-test.hybrid-multi.trec
[Skip] Existing TREC run found: runs\run.dpr.curated-test.multi.trec
[Skip] Existing Hybrid run found: runs\run.hybrid.curated-test.hybrid-multi.trec
[Skip] Existing TREC run found: runs\run.dpr.squad-test.multi.trec
[Skip] Existing Hybrid run found: runs\run.hybrid.squad-test.hybrid-multi.trec

=== dpr-single-nq ===
[Skip] Existing TREC run found: runs\run.dpr.nq-test.dpr-single-nq.trec
[Skip] Existing Hybrid run found: runs\run.hybrid.nq-test.hybrid-single-nq.trec

Running 12 convert+evaluate jobs with 3 in parallel...
>> c:\Users\Dwija Patel\dwija\

,Dataset,Retriever,Index,Top-20,Top-100
0,NQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.94,78.25
1,TRIVIA,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,76.41,83.15
2,WQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.40,75.49
3,CURATED,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,80.69,89.91
4,SQUAD,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,71.09,81.84
5,NQ,DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,79.47,86.09
6,NQ,BM25 + DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,81.75,87.62
7,TRIVIA,DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,78.87,84.79
8,TRIVIA,BM25 + DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,81.74,85.95
9,WQ,DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,75.05,82.97


Top-20                             Top-100         \
Dataset                CURATED     NQ  SQUAD TRIVIA     WQ CURATED     NQ   
Retriever                                                                   
BM25                     80.69  62.94  71.09  76.41  62.40   89.91  78.25   
BM25 + DPR-Multi         89.91  81.75  71.20  81.74  76.53   94.09  87.62   
BM25 + DPR-Single (NQ)     NaN  82.77    NaN    NaN    NaN     NaN  87.98   
DPR-Multi                88.76  79.47  51.99  78.87  75.05   93.37  86.09   
DPR-Single (NQ)            NaN  80.06    NaN    NaN    NaN     NaN  86.09   

                                             
Dataset                 SQUAD TRIVIA     WQ  
Retriever                                    
BM25                    81.84  83.15  75.49  
BM25 + DPR-Multi        82.58  85.95  84.06  
BM25 + DPR-Single (NQ)    NaN    NaN    NaN  
DPR-Multi               67.73  84.79  82.97  
DPR-Single (NQ)           NaN    NaN    NaN

## 7. Extension 1 - RRF

In [23]:
# ===== Shared helpers for Extensions 1 & 2 (define ONCE; do not redefine in later cells) =====
import ast, json, itertools, random
from pathlib import Path
import pandas as pd
from pyserini.search import get_topics
from pyserini.search.lucene import LuceneSearcher
from pyserini.eval.evaluate_dpr_retrieval import has_answers, SimpleTokenizer

lucene_searcher = LuceneSearcher(root)
simple_tok = SimpleTokenizer()
passage_cache = {}   # docid -> raw "contents" string (this name is used for nothing else)

def load_trec_topk(trec_path, max_k=100):
    runs = {}
    with open(trec_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                qid, _, docid, _, score, _ = parts[:6]
                lst = runs.setdefault(qid, [])
                if len(lst) < max_k:
                    lst.append((docid, float(score)))
    return runs

def get_cached_passage(docid):
    docid = str(docid)
    if docid not in passage_cache:
        doc = lucene_searcher.doc(docid)
        passage_cache[docid] = json.loads(doc.raw()).get('contents', '') if doc else ''
    return passage_cache[docid]

def parse_answers(raw):
    if isinstance(raw, list):
        return [str(a) for a in raw]
    if isinstance(raw, str) and raw.startswith('['):
        try:
            return [str(a) for a in ast.literal_eval(raw)]
        except Exception:
            pass
    return [str(raw)]

def eval_recall_and_mrr(runs, topics, use_regex=False, qids_subset=None, ks=(5, 20, 100), depth=100):
    # Recall@k and MRR of the first answer-containing passage within `depth`. Query ids are always str.
    tb = {str(k): v for k, v in topics.items()}
    qids = [str(q) for q in qids_subset] if qids_subset is not None else [q for q in tb if q in runs]
    hits = {k: 0 for k in ks}
    mrr_sum = 0.0
    for qid in qids:
        answers = parse_answers(tb[qid]['answers'])
        first = None
        for rank, (docid, _) in enumerate(runs.get(qid, [])[:depth], start=1):
            if has_answers(get_cached_passage(docid), answers, simple_tok, regex=use_regex):
                first = rank
                break
        if first is not None:
            for k in ks:
                if first <= k:
                    hits[k] += 1
            mrr_sum += 1.0 / first
    n = max(len(qids), 1)
    out = {f'Recall@{k}': round(100.0 * hits[k] / n, 2) for k in ks}
    out['MRR'] = round(mrr_sum / n, 4)
    return out

# ---- ONE tune/eval split per dataset, saved to disk and reused by Extensions 1 and 2 ----
SPLIT_PATH = RUNS_DIR / 'splits.json'

def get_splits(force=False):
    if SPLIT_PATH.exists() and not force:
        with open(SPLIT_PATH, encoding='utf-8') as f:
            return json.load(f)
    out = {}
    for ds_key, ds_cfg in ALL_DATASETS.items():
        qids = [str(q) for q in get_topics(ds_cfg['topics'])]
        random.Random(0).shuffle(qids)
        n_tune = min(300, max(100, int(0.2 * len(qids))))
        out[ds_key] = {'tune': qids[:n_tune], 'eval': qids[n_tune:]}
    with open(SPLIT_PATH, 'w', encoding='utf-8') as f:
        json.dump(out, f)
    return out

splits = get_splits()
print({k: (len(v['tune']), len(v['eval'])) for k, v in splits.items()})

{'nq': (300, 3310), 'trivia': (300, 11013), 'wq': (300, 1732), 'curated': (138, 556), 'squad': (300, 10270)}


In [24]:
# ===== Extension 1: RRF vs. linear fusion (tuned on the tune split, reported on the held-out eval split) =====
from tqdm.auto import tqdm

def fuse_rrf(bm25_runs, dpr_runs, k=60, w_bm25=1.0, depth=100, topk=100, qids=None):
    fused = {}
    for qid in (qids if qids is not None else dpr_runs):
        sc = {}
        for r, (d, _) in enumerate(bm25_runs.get(qid, [])[:depth], start=1):
            sc[d] = sc.get(d, 0.0) + w_bm25 / (k + r)
        for r, (d, _) in enumerate(dpr_runs.get(qid, [])[:depth], start=1):
            sc[d] = sc.get(d, 0.0) + 1.0 / (k + r)
        fused[qid] = sorted(sc.items(), key=lambda x: (-x[1], x[0]))[:topk]
    return fused

def fuse_linear(bm25_runs, dpr_runs, lam, topk=100, qids=None):
    # Same convention as the paper and as build_hybrid_trec_run in section 6:  BM25 + lam * dense
    out = {}
    for qid in (qids if qids is not None else dpr_runs):
        d, b = dict(dpr_runs[qid]), dict(bm25_runs.get(qid, []))
        d_min = min(d.values())
        b_min = min(b.values()) if b else 0.0
        sc = {x: b.get(x, b_min) + lam * d.get(x, d_min) for x in set(d) | set(b)}
        out[qid] = sorted(sc.items(), key=lambda t: (-t[1], t[0]))[:topk]
    return out

RRF_GRID = list(itertools.product([10, 20, 60], [0.5, 0.75, 1.0, 1.5, 2.0], [100, 1000]))   # (k, w_bm25, depth)
LAM_GRID = [0.5, 0.8, 1.1, 1.5, 2.0, 3.0]                                                       # includes paper's 1.1

def tune_and_eval(d, dpr_runs, dpr_label, paper_linear_run, desc='tune'):
    # d needs: topics, regex, bm25, tune_qids, eval_qids (all qids are str)
    tune, ev = d['tune_qids'], d['eval_qids']
    kw = dict(topics=d['topics'], use_regex=d['regex'])
    score = lambda m: (m['Recall@20'], m['Recall@100'])

    total_steps = len(RRF_GRID) + len(LAM_GRID) + 1          # + final eval
    with tqdm(total=total_steps, desc=desc, leave=False) as pbar:
        best_rrf, best_rrf_s = None, (-1, -1)
        for k_val, w, depth in RRF_GRID:
            pbar.set_postfix_str(f'RRF k={k_val} w={w} depth={depth}')
            m = eval_recall_and_mrr(fuse_rrf(d['bm25'], dpr_runs, k_val, w, depth, 100, tune), qids_subset=tune, **kw)
            if score(m) > best_rrf_s:
                best_rrf_s, best_rrf = score(m), dict(k=k_val, w_bm25=w, depth=depth)
            pbar.update(1)

        best_lam, best_lam_s = None, (-1, -1)
        for lam in LAM_GRID:
            pbar.set_postfix_str(f'linear lam={lam}')
            m = eval_recall_and_mrr(fuse_linear(d['bm25'], dpr_runs, lam, 100, tune), qids_subset=tune, **kw)
            if score(m) > best_lam_s:
                best_lam_s, best_lam = score(m), lam
            pbar.update(1)

        pbar.set_postfix_str('final eval on held-out split')
        runs = [
            (dpr_label, dpr_runs),
            ('BM25', d['bm25']),
            ('Linear (λ=1.1, paper)', paper_linear_run),
            (f'Linear (tuned λ={best_lam})', fuse_linear(d['bm25'], dpr_runs, best_lam, 100, ev)),
            (f'RRF (tuned {best_rrf})', fuse_rrf(d['bm25'], dpr_runs, qids=ev, **best_rrf)),
        ]
        rows = [{'Method': lbl, **eval_recall_and_mrr(r, qids_subset=ev, **kw)} for lbl, r in runs]
        pbar.update(1)
    return rows, best_rrf, best_lam

ext1_rows, best_cfg = [], {}
outer = tqdm(ALL_DATASETS.items(), total=len(ALL_DATASETS), desc='Extension 1 (datasets)')
for ds_key, ds_cfg in outer:
    outer.set_postfix_str(f'now: {ds_key}')
    tag = ds_cfg['tag']
    topics = get_topics(ds_cfg['topics'])
    dpr_r  = load_trec_topk(RUNS_DIR / f'run.dpr.{tag}.multi.trec', max_k=1000)
    bm25_r = load_trec_topk(RUNS_DIR / f'run.dpr.{tag}.bm25.trec', max_k=1000)
    lin_r  = load_trec_topk(RUNS_DIR / f'run.hybrid.{tag}.hybrid-multi.trec', max_k=100)
    d = dict(topics=topics, regex=ds_cfg['regex'], bm25=bm25_r,
             tune_qids=splits[ds_key]['tune'], eval_qids=splits[ds_key]['eval'])

    rows, rrf_cfg, lam = tune_and_eval(d, dpr_r, 'DPR-Multi', lin_r, desc=f'{ds_key}: tuning')
    best_cfg[ds_key] = dict(rrf=rrf_cfg, lam=lam)
    ext1_rows += [{'Dataset': ds_key.upper(), **r} for r in rows]

    # save the full tuned-RRF run (all queries)
    full = fuse_rrf(bm25_r, dpr_r, **rrf_cfg)
    with open(RUNS_DIR / f'run.dpr.{tag}.rrf-tuned.trec', 'w', encoding='utf-8') as f:
        for qid, docs in full.items():
            for rank, (docid, sc) in enumerate(docs, start=1):
                f.write(f'{qid} Q0 {docid} {rank} {sc:.8f} rrf-tuned\n')

ext1_df = pd.DataFrame(ext1_rows)
ext1_df.to_csv(RUNS_DIR / 'extension1_rrf_heldout_metrics.csv', index=False)
with open(RUNS_DIR / 'extension1_best_cfg.json', 'w', encoding='utf-8') as f:
    json.dump(best_cfg, f, indent=2)
display(best_cfg)
display(ext1_df)

Extension 1 (datasets):   0%|          | 0/5 [00:00<?, ?it/s]

nq: tuning:   0%|          | 0/37 [00:00<?, ?it/s]

trivia: tuning:   0%|          | 0/37 [00:00<?, ?it/s]

wq: tuning:   0%|          | 0/37 [00:00<?, ?it/s]

curated: tuning:   0%|          | 0/37 [00:00<?, ?it/s]

squad: tuning:   0%|          | 0/37 [00:00<?, ?it/s]

{'nq': {'rrf': {'k': 60, 'w_bm25': 1.0, 'depth': 1000}, 'lam': 0.8},
 'trivia': {'rrf': {'k': 10, 'w_bm25': 1.0, 'depth': 100}, 'lam': 2.0},
 'wq': {'rrf': {'k': 10, 'w_bm25': 0.5, 'depth': 100}, 'lam': 1.5},
 'curated': {'rrf': {'k': 60, 'w_bm25': 0.75, 'depth': 1000}, 'lam': 2.0},
 'squad': {'rrf': {'k': 20, 'w_bm25': 1.0, 'depth': 1000}, 'lam': 0.5}}

,Dataset,Method,Recall@5,Recall@20,Recall@100,MRR
0,NQ,DPR-Multi,67.55,79.67,86.19,0.5485
1,NQ,BM25,44.29,63.53,78.52,0.3276
2,NQ,"Linear (λ=1.1, paper)",71.39,81.81,87.76,0.5895
3,NQ,Linear (tuned λ=0.8),69.70,80.57,87.37,0.5751
4,NQ,"RRF (tuned {'k': 60, 'w_bm25': 1.0, 'depth': 1...",68.07,81.93,88.19,0.5420
5,TRIVIA,DPR-Multi,70.10,79.09,84.92,0.6080
6,TRIVIA,BM25,67.31,77.20,83.80,0.5674
7,TRIVIA,"Linear (λ=1.1, paper)",75.91,82.32,86.41,0.6814
8,TRIVIA,Linear (tuned λ=2.0),75.72,82.87,86.75,0.6759
9,TRIVIA,"RRF (tuned {'k': 10, 'w_bm25': 1.0, 'depth': 1...",75.43,82.52,86.82,0.6636


In [25]:
# ===== Extension 1 on DPR-Single (NQ): tuned on its OWN tune split, reported on the same held-out NQ eval split =====
nq_cfg = ALL_DATASETS['nq']
nq_topics = get_topics(nq_cfg['topics'])
bm25_nq       = load_trec_topk(RUNS_DIR / 'run.dpr.nq-test.bm25.trec', max_k=1000)
dpr_single_nq = load_trec_topk(RUNS_DIR / 'run.dpr.nq-test.dpr-single-nq.trec', max_k=1000)
lin_single_nq = load_trec_topk(RUNS_DIR / 'run.hybrid.nq-test.hybrid-single-nq.trec', max_k=100)

d_nq = dict(topics=nq_topics, regex=False, bm25=bm25_nq,
            tune_qids=splits['nq']['tune'], eval_qids=splits['nq']['eval'])
rows, rrf_cfg_single, lam_single = tune_and_eval(d_nq, dpr_single_nq, 'DPR-Single (NQ)', lin_single_nq)
print('DPR-Single (NQ) tuned:', rrf_cfg_single, 'lambda =', lam_single)

rrf_single_full = fuse_rrf(bm25_nq, dpr_single_nq, **rrf_cfg_single)
with open(RUNS_DIR / 'run.dpr.nq-test.rrf-single-nq-tuned.trec', 'w', encoding='utf-8') as f:
    for qid, docs in rrf_single_full.items():
        for rank, (docid, sc) in enumerate(docs, start=1):
            f.write(f'{qid} Q0 {docid} {rank} {sc:.8f} rrf-single-nq-tuned\n')

single_final_df = pd.DataFrame([{'Dataset': 'NQ', **r} for r in rows])
single_final_df.to_csv(RUNS_DIR / 'extension1_rrf_single_nq_metrics.csv', index=False)
display(single_final_df)

tune:   0%|          | 0/37 [00:00<?, ?it/s]

DPR-Single (NQ) tuned: {'k': 60, 'w_bm25': 1.0, 'depth': 1000} lambda = 0.8


,Dataset,Method,Recall@5,Recall@20,Recall@100,MRR
0,NQ,DPR-Single (NQ),68.55,80.21,86.28,0.5642
1,NQ,BM25,44.29,63.53,78.52,0.3276
2,NQ,"Linear (λ=1.1, paper)",72.27,82.81,88.10,0.5992
3,NQ,Linear (tuned λ=0.8),71.81,82.96,88.58,0.5951
4,NQ,"RRF (tuned {'k': 60, 'w_bm25': 1.0, 'depth': 1...",68.61,82.05,88.37,0.5363


## 8. Extension 2 - Post Retreival Cross Encoder

In [26]:
# ===== Extension 2: cross-encoder reranking of DPR (uses the shared helpers + splits from section 7) =====
import time, warnings
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore', category=DeprecationWarning, message='.*torch.ao.quantization.*')

CE_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
RERANK_TOP_M = 50
CE_BATCH_SIZE = 64
MAX_EVAL_QUERIES = 500      # taken from the held-out eval split; the next cell tunes on the first 200 and reports on the rest
KS = (5, 20)                # Recall@50 is omitted: reranking inside the top-50 cannot change it

ce_tokenizer = AutoTokenizer.from_pretrained(CE_MODEL_NAME)
ce_model = AutoModelForSequenceClassification.from_pretrained(CE_MODEL_NAME).eval()
if DEVICE == 'cpu':
    ce_model = torch.quantization.quantize_dynamic(ce_model, {torch.nn.Linear}, dtype=torch.qint8)
else:
    ce_model = ce_model.to(DEVICE)
print(f'Loaded {CE_MODEL_NAME} on {DEVICE} (top_m={RERANK_TOP_M}, max_queries={MAX_EVAL_QUERIES})')

@torch.no_grad()
def rerank_with_cross_encoder(dpr_runs, topics, qids, out_trec_path, top_m=RERANK_TOP_M,
                              batch_size=CE_BATCH_SIZE, run_tag='dpr-ce'):
    tb = {str(k): v for k, v in topics.items()}
    reranked, total_sec = {}, 0.0
    with open(out_trec_path, 'w', encoding='utf-8') as fout:
        for qid in tqdm(qids, desc=f'CE Rerank ({run_tag})'):
            q_text = tb[qid]['title']
            docids = [d for d, _ in dpr_runs[qid][:top_m]]
            passages = [get_cached_passage(d) for d in docids]      # Lucene lookup is NOT included in the latency

            t0 = time.perf_counter()
            ce_scores = []
            for i in range(0, len(passages), batch_size):
                b = passages[i:i + batch_size]
                enc = ce_tokenizer([q_text] * len(b), b, padding=True, truncation=True,
                                   max_length=192, return_tensors='pt').to(DEVICE)
                logits = ce_model(**enc).logits.squeeze(-1)
                if logits.ndim == 0:
                    logits = logits.unsqueeze(0)
                ce_scores.extend(logits.cpu().tolist())
            if DEVICE == 'cuda':
                torch.cuda.synchronize()
            total_sec += time.perf_counter() - t0

            ranked = sorted(zip(docids, ce_scores), key=lambda x: x[1], reverse=True)
            reranked[qid] = ranked
            for rank, (docid, score) in enumerate(ranked, start=1):
                fout.write(f'{qid} Q0 {docid} {rank} {score:.6f} {run_tag}\n')
    return reranked, round(1000.0 * total_sec / max(len(qids), 1), 2)

ce_tasks = [('DPR-Multi', ds_key, ALL_DATASETS[ds_key],
             RUNS_DIR / f"run.dpr.{ALL_DATASETS[ds_key]['tag']}.multi.trec",
             RUNS_DIR / f"run.dpr.{ALL_DATASETS[ds_key]['tag']}.dpr-multi-ce-minilm-l6.trec")
            for ds_key in ALL_DATASETS]
ce_tasks.append(('DPR-Single (NQ)', 'nq', ALL_DATASETS['nq'],
                 RUNS_DIR / 'run.dpr.nq-test.dpr-single-nq.trec',
                 RUNS_DIR / 'run.dpr.nq-test.dpr-single-nq-ce-minilm-l6.trec'))

ce_eval_qids = {}   # ds_key -> the exact queries CE was run on (reused in the next cell)
ce_compare_rows = []
for label, ds_key, ds_cfg, dpr_path, ce_path in ce_tasks:
    topics = get_topics(ds_cfg['topics'])
    qids = splits[ds_key]['eval'][:MAX_EVAL_QUERIES]
    ce_eval_qids[ds_key] = qids
    orig = load_trec_topk(dpr_path, max_k=RERANK_TOP_M)
    reranked, lat_ms = rerank_with_cross_encoder(orig, topics, qids, ce_path, run_tag=f'{ds_key}-{label}-ce')
    kw = dict(use_regex=ds_cfg['regex'], qids_subset=qids, ks=KS, depth=RERANK_TOP_M)
    ce_compare_rows.append({'Dataset': ds_key.upper(), 'Method': f'{label} (Original)',
                            'Rerank Latency (ms/q)': 0.0, **eval_recall_and_mrr(orig, topics, **kw)})
    ce_compare_rows.append({'Dataset': ds_key.upper(), 'Method': f'{label} + Cross-Encoder (MiniLM-L6)',
                            'Rerank Latency (ms/q)': lat_ms, **eval_recall_and_mrr(reranked, topics, **kw)})

del ce_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

ce_compare_df = pd.DataFrame(ce_compare_rows)
ce_compare_df.to_csv(RUNS_DIR / 'extension2_dpr_only_ce_comparison.csv', index=False)
display(ce_compare_df)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Lib\site-packages\torch\ao\nn\quantized\modules\utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  qweight = torch.quantize_per_tensor(


Loaded cross-encoder/ms-marco-MiniLM-L-6-v2 on cpu (top_m=50, max_queries=500)


CE Rerank (nq-DPR-Multi-ce):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (trivia-DPR-Multi-ce):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (wq-DPR-Multi-ce):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (curated-DPR-Multi-ce):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (squad-DPR-Multi-ce):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (nq-DPR-Single (NQ)-ce):   0%|          | 0/500 [00:00<?, ?it/s]

,Dataset,Method,Rerank Latency (ms/q),Recall@5,Recall@20,MRR
0,NQ,DPR-Multi (Original),0.00,70.0,81.4,0.5539
1,NQ,DPR-Multi + Cross-Encoder (MiniLM-L6),1814.65,74.4,83.4,0.6071
2,TRIVIA,DPR-Multi (Original),0.00,70.2,78.2,0.6026
3,TRIVIA,DPR-Multi + Cross-Encoder (MiniLM-L6),1727.46,75.8,81.2,0.6831
4,WQ,DPR-Multi (Original),0.00,65.8,75.0,0.5431
5,WQ,DPR-Multi + Cross-Encoder (MiniLM-L6),1429.53,69.2,79.4,0.5567
6,CURATED,DPR-Multi (Original),0.00,81.4,89.2,0.7225
7,CURATED,DPR-Multi + Cross-Encoder (MiniLM-L6),1439.98,88.0,92.2,0.7853
8,SQUAD,DPR-Multi (Original),0.00,35.2,53.2,0.2432
9,SQUAD,DPR-Multi + Cross-Encoder (MiniLM-L6),1390.40,51.0,59.4,0.4320


In [31]:
# --- reset a possibly polluted passage cache and make the helper type-safe ---
passage_cache.clear()

def get_cached_passage(docid):
    docid = str(docid)
    if docid not in passage_cache:
        doc = lucene_searcher.doc(docid)
        text = json.loads(doc.raw()).get('contents', '') if doc else ''
        if isinstance(text, (tuple, list)):      # defensive: never cache non-strings
            text = ' '.join(map(str, text))
        passage_cache[docid] = str(text)
    return passage_cache[docid]

# ===== Extension 2b (ALL datasets): blend DPR and CE scores. Per (retriever, dataset): tune (M, beta) on the first 200 CE-scored queries, report on the rest =====
def minmax_norm(scores):
    lo, hi = min(scores), max(scores)
    return [0.5] * len(scores) if (hi - lo) < 1e-9 else [(s - lo) / (hi - lo) for s in scores]

def blend_dpr_and_ce(dpr_runs, ce_runs, beta=0.8, top_m=50):
    out = {}
    for qid, dpr_cands in dpr_runs.items():
        if qid not in ce_runs:          # only queries that were actually CE-scored
            continue
        head, tail = dpr_cands[:top_m], dpr_cands[top_m:]
        ce_map = dict(ce_runs[qid])
        docids = [d for d, _ in head]
        dpr_norm = minmax_norm([sc for _, sc in head])
        ce_norm  = minmax_norm([ce_map.get(d, 0.0) for d in docids])
        rescored = sorted(((d, (1.0 - beta) * a + beta * b) for d, a, b in zip(docids, dpr_norm, ce_norm)),
                          key=lambda x: x[1], reverse=True)
        min_sc = rescored[-1][1] if rescored else 0.0
        out[qid] = rescored + [(d, min_sc - 1.0 - i) for i, (d, _) in enumerate(tail)]
    return out

N_TUNE_CE = 200
M_GRID, BETA_GRID = [10, 20, 50], [0.0, 0.5, 0.7, 0.85, 1.0]

# (label, dataset key, dpr run file, ce run file)
blend_tasks = [('DPR-Multi', ds_key,
                f"run.dpr.{ALL_DATASETS[ds_key]['tag']}.multi.trec",
                f"run.dpr.{ALL_DATASETS[ds_key]['tag']}.dpr-multi-ce-minilm-l6.trec")
               for ds_key in ALL_DATASETS]
blend_tasks.append(('DPR-Single (NQ)', 'nq', 'run.dpr.nq-test.dpr-single-nq.trec',
                    'run.dpr.nq-test.dpr-single-nq-ce-minilm-l6.trec'))

def _lat(ds_key, label):
    r = ce_compare_df[(ce_compare_df['Dataset'] == ds_key.upper()) &
                      (ce_compare_df['Method'] == f'{label} + Cross-Encoder (MiniLM-L6)')]
    return float(r['Rerank Latency (ms/q)'].iloc[0])

tune_all, final_all, best_blend = [], [], {}
for label, ds_key, dpr_file, ce_file in tqdm(blend_tasks, desc='Extension 2b'):
    ds_cfg = ALL_DATASETS[ds_key]
    topics = get_topics(ds_cfg['topics'])
    kw = dict(use_regex=ds_cfg['regex'], ks=KS, depth=RERANK_TOP_M)
    qids = ce_eval_qids[ds_key]
    tune_ids, report_ids = qids[:N_TUNE_CE], qids[N_TUNE_CE:]

    dpr_r = load_trec_topk(RUNS_DIR / dpr_file, max_k=RERANK_TOP_M)
    ce_r  = load_trec_topk(RUNS_DIR / ce_file,  max_k=RERANK_TOP_M)   # already in CE-ranked order

    rows = []
    for m_val in M_GRID:
        for beta_val in BETA_GRID:
            met = eval_recall_and_mrr(blend_dpr_and_ce(dpr_r, ce_r, beta_val, m_val), topics,
                                      qids_subset=tune_ids, **kw)
            rows.append({'Retriever': label, 'Dataset': ds_key.upper(), 'Top-M': m_val,
                         'Beta (CE weight)': beta_val, **met})
    tdf = pd.DataFrame(rows).sort_values(by=['MRR', 'Recall@20'], ascending=False).reset_index(drop=True)
    tune_all.append(tdf)
    best_m, best_beta = int(tdf.loc[0, 'Top-M']), float(tdf.loc[0, 'Beta (CE weight)'])
    best_blend[f'{label}|{ds_key}'] = dict(M=best_m, beta=best_beta)
    print(f'{ds_key.upper():8s} {label:16s} -> tune on {len(tune_ids)}, report on {len(report_ids)} | M={best_m}, beta={best_beta}')

    ce_lat = _lat(ds_key, label)
    blended = blend_dpr_and_ce(dpr_r, ce_r, best_beta, best_m)
    for name, run, lat in [(label, dpr_r, 0.0),
                           (f'{label} + CE only', ce_r, ce_lat),
                           (f'{label} + blend (M={best_m}, beta={best_beta})', blended,
                            round(ce_lat * best_m / RERANK_TOP_M, 1))]:
        final_all.append({'Dataset': ds_key.upper(), 'Method': name, 'Est. latency (ms/q)': lat,
                          **eval_recall_and_mrr(run, topics, qids_subset=report_ids, **kw)})

ce_tune_all_df = pd.concat(tune_all, ignore_index=True)
ext2_final_all_df = pd.DataFrame(final_all)
ce_tune_all_df.to_csv(RUNS_DIR / 'extension2_blend_tuning_all.csv', index=False)
ext2_final_all_df.to_csv(RUNS_DIR / 'extension2_blend_heldout_metrics_all.csv', index=False)
with open(RUNS_DIR / 'extension2_best_blend_cfg.json', 'w', encoding='utf-8') as f:
    json.dump(best_blend, f, indent=2)
display(ext2_final_all_df)

Extension 2b:   0%|          | 0/6 [00:00<?, ?it/s]

NQ       DPR-Multi        -> tune on 200, report on 300 | M=10, beta=0.85
TRIVIA   DPR-Multi        -> tune on 200, report on 300 | M=50, beta=0.85
WQ       DPR-Multi        -> tune on 200, report on 300 | M=20, beta=0.5
CURATED  DPR-Multi        -> tune on 200, report on 300 | M=50, beta=0.85
SQUAD    DPR-Multi        -> tune on 200, report on 300 | M=50, beta=0.85
NQ       DPR-Single (NQ)  -> tune on 200, report on 300 | M=50, beta=0.85


,Dataset,Method,Est. latency (ms/q),Recall@5,Recall@20,MRR
0,NQ,DPR-Multi,0.00,72.00,82.33,0.5651
1,NQ,DPR-Multi + CE only,1814.65,76.67,84.00,0.6346
2,NQ,"DPR-Multi + blend (M=10, beta=0.85)",362.90,76.33,82.33,0.6440
3,TRIVIA,DPR-Multi,0.00,69.67,78.67,0.6003
4,TRIVIA,DPR-Multi + CE only,1727.46,76.33,81.67,0.6874
5,TRIVIA,"DPR-Multi + blend (M=50, beta=0.85)",1727.50,75.67,81.00,0.6901
6,WQ,DPR-Multi,0.00,66.33,74.67,0.5329
7,WQ,DPR-Multi + CE only,1429.53,69.67,79.33,0.5581
8,WQ,"DPR-Multi + blend (M=20, beta=0.5)",571.80,67.67,74.67,0.5806
9,CURATED,DPR-Multi,0.00,81.33,88.33,0.7142
